# Atensi dari Nol hingga Encoder Transformer

**Buku:** *Deep Learning dengan Python* — Bab 8, Atensi dan Transformer

Menghitung *scaled dot-product attention* secara manual, lalu membangun encoder Transformer untuk klasifikasi sentimen IMDB dan melihat bobot atensinya.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±10 menit

**Isi notebook:**
- Atensi dengan `keras.ops`: $\mathrm{softmax}(QK^\top/\sqrt{d_k})\,V$.
- Lapisan `EmbeddingPosisi` dan `EncoderTransformer` buatan sendiri (bisa disimpan/dimuat).
- Klasifikasi IMDB dan perbandingan dengan model *bag-of-words*.
- Visualisasi peta atensi pada ulasan sungguhan.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. *Scaled dot-product attention* secara manual

In [ ]:
import numpy as np
from keras import ops

def atensi(Q, K, V, mask=None):
    d_k = ops.shape(K)[-1]
    skor = ops.matmul(Q, ops.transpose(K)) / ops.sqrt(ops.cast(d_k, "float32"))
    if mask is not None:
        skor = ops.where(mask, skor, -1e9)
    bobot = ops.softmax(skor, axis=-1)
    return ops.matmul(bobot, V), bobot

rng = np.random.default_rng(0)
kata = ["kucing", "itu", "duduk", "di", "tikar"]
X = rng.normal(size=(5, 8)).astype("float32")             # 5 token, dimensi 8
Wq, Wk, Wv = (rng.normal(size=(8, 8)).astype("float32") for _ in range(3))
keluaran, bobot = atensi(X @ Wq, X @ Wk, X @ Wv)
print("Keluaran:", keluaran.shape)
print("Jumlah tiap baris bobot:", ops.convert_to_numpy(ops.sum(bobot, axis=-1)))

In [ ]:
import matplotlib.pyplot as plt

def gambar_peta(bobot, label_q, label_k, judul=""):
    plt.figure(figsize=(0.45 * len(label_k) + 2, 0.45 * len(label_q) + 1.5))
    plt.imshow(ops.convert_to_numpy(bobot), cmap="Blues")
    plt.xticks(range(len(label_k)), label_k, rotation=90); plt.yticks(range(len(label_q)), label_q)
    plt.xlabel("Kunci (key)"); plt.ylabel("Kueri (query)"); plt.title(judul); plt.colorbar(); plt.show()

gambar_peta(bobot, kata, kata, "Bobot atensi (proyeksi acak)")

# Masker kausal: token hanya boleh melihat dirinya dan token sebelumnya
kausal = np.tril(np.ones((5, 5), dtype=bool))
_, bobot_kausal = atensi(X @ Wq, X @ Wk, X @ Wv, mask=kausal)
gambar_peta(bobot_kausal, kata, kata, "Dengan masker kausal")

## 2. Lapisan Transformer

In [ ]:
from keras import layers, ops

class EmbeddingPosisi(layers.Layer):
    """Embedding token + embedding posisi yang dipelajari."""
    def __init__(self, panjang, kosakata, dim, **kwargs):
        super().__init__(**kwargs)
        self.token = layers.Embedding(kosakata, dim)
        self.posisi = layers.Embedding(panjang, dim)
        self.cfg = dict(panjang=panjang, kosakata=kosakata, dim=dim)

    def call(self, x):
        pos = ops.arange(0, ops.shape(x)[-1])
        return self.token(x) + self.posisi(pos)

    def get_config(self):
        return {**super().get_config(), **self.cfg}


class EncoderTransformer(layers.Layer):
    """Satu blok encoder: self-attention + FFN, masing-masing dengan residual dan LayerNorm."""
    def __init__(self, dim, dim_ff, n_head, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.atensi = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head)
        self.ffn = keras.Sequential([layers.Dense(dim_ff, activation="gelu"), layers.Dense(dim)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.cfg = dict(dim=dim, dim_ff=dim_ff, n_head=n_head, dropout=dropout)

    def call(self, x, token=None, training=False):
        mask = None
        if token is not None:                                  # abaikan posisi padding (token 0)
            mask = ops.expand_dims(ops.not_equal(token, 0), 1)
        a = self.atensi(x, x, attention_mask=mask, training=training)
        x = self.norm1(x + self.drop1(a, training=training))
        return self.norm2(x + self.drop2(self.ffn(x), training=training))

    def get_config(self):
        return {**super().get_config(), **self.cfg}

## 3. Data IMDB

In [ ]:
from keras.datasets import imdb

KOSAKATA, PANJANG = 20000, 256
(x_latih, y_latih), (x_uji, y_uji) = imdb.load_data(num_words=KOSAKATA)
x_latih = keras.utils.pad_sequences(x_latih, maxlen=PANJANG)
x_uji = keras.utils.pad_sequences(x_uji, maxlen=PANJANG)
x_latih, x_val = x_latih[5000:], x_latih[:5000]
y_latih, y_val = y_latih[5000:], y_latih[:5000]

## 4. Baseline *bag-of-words* (tanpa urutan)

In [ ]:
bow = keras.Sequential([keras.Input(shape=(PANJANG,), dtype="int32"),
                        layers.Embedding(KOSAKATA, 64),
                        layers.GlobalAveragePooling1D(),
                        layers.Dense(1, activation="sigmoid")])
bow.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bow.fit(x_latih, y_latih, epochs=5, batch_size=64, validation_data=(x_val, y_val))
print(f"Bag-of-words, akurasi uji: {bow.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")

## 5. Klasifikasi dengan encoder Transformer

In [ ]:
DIM, DIM_FF, N_HEAD = 64, 128, 4

masukan = keras.Input(shape=(PANJANG,), dtype="int32")
x = EmbeddingPosisi(PANJANG, KOSAKATA, DIM, name="embedding_posisi")(masukan)
x = EncoderTransformer(DIM, DIM_FF, N_HEAD, name="encoder")(x, masukan)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.3)(x)
keluaran = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(masukan, keluaran)
model.summary()

model.compile(optimizer=keras.optimizers.Adam(3e-4), loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_latih, y_latih, epochs=5, batch_size=64, validation_data=(x_val, y_val),
          callbacks=[keras.callbacks.ModelCheckpoint("transformer_imdb.keras", save_best_only=True)])

In [ ]:
# Lapisan kustom harus didaftarkan saat memuat model
model = keras.models.load_model("transformer_imdb.keras",
    custom_objects={"EmbeddingPosisi": EmbeddingPosisi, "EncoderTransformer": EncoderTransformer})
print(f"Transformer, akurasi uji: {model.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")

Pada IMDB, Transformer kecil biasanya hanya sedikit lebih baik dari *bag-of-words*. Sentimen banyak ditentukan oleh kata kunci, bukan urutan. Keunggulan Transformer terlihat jelas pada tugas yang bergantung pada urutan, seperti penerjemahan di notebook berikutnya.

## 6. Peta atensi pada ulasan sungguhan

In [ ]:
indeks_kata = imdb.get_word_index()
ke_kata = {i + 3: k for k, i in indeks_kata.items()}
ke_kata.update({0: "<pad>", 1: "<mulai>", 2: "<unk>"})

contoh = x_uji[:1]
token = [ke_kata.get(int(t), "?") for t in contoh[0]]
awal = int(np.argmax(contoh[0] != 0))             # lewati padding di depan
n = 25

emb = model.get_layer("embedding_posisi")(contoh)
_, skor = model.get_layer("encoder").atensi(emb, emb, return_attention_scores=True)
skor = ops.convert_to_numpy(skor)[0].mean(axis=0)      # rata-rata semua head
label = token[awal:awal + n]
gambar_peta(skor[awal:awal + n, awal:awal + n], label, label,
            f"Atensi rata-rata head (label: {'positif' if y_uji[0] else 'negatif'})")